TOKENIZZAZIONE CON NLTK E SPACY

Approfondiamo il concetto di tokenizzazione.

Modelli Linguistici e Inizzializzazione
Preparare l'ambiente per l'elaborazione
Entrare nel mondo del NLP moderno significa smettere di trattare il testo come una banale sequenza di caratteri e iniziare a trattarlo come un oggetto linguistico vivo e strutturato. Per fare questo, libreria come spaCy, non si limitano a "tagliare" le parolo, esso carica modelli statistici pre-addestr (che ha già letto miliardi di frasi) che contengono pesi neurali e vocabolari specifici per una lingua.
Il caricamente di un modello come "it_core_news_sm" non è una semplice importazione di codice, ma l'attivazione di una pipeline complessa che include pesi per il training, il parsing, e, appunto, la tokenizzazione avanzata, è come assumere un'assistente che conosce già le basi della grammatica italiana.

Ma non tutti i modelli sono uguali

Vediamo le differenze
Ecositema dei Modelli spaCy
- Modelli SM (Smal): ottimizzati per la velocità e il risparmio di memoria, ideali per tokenizzazione veloce e taggin senza dipendenze pesanti. Perfetti se abbiamo poca memoria o ci serve una tokenizzazione rapida
- Modelli MD/LG (medium/large): includono vettori di parole (word vectors) densi, permettendo confronti semantici più accurati tra i token. E' come passare da un vocabolario tascabile ad uno più ricco.
- Piattaforma Multilingue: gestione dei modelli tramite il comando "spacy download" che sincronizza i pesi locali con il reposity ufficiale
- Lazy Loading vs Eager Loading: la differenza tra inizializare la pipeline al volo o caricarla internamente in RAM all'avvio del sistema. Preferire caricare tutto subito (Eager) se l'app deve rispondere all'istante all'utente, altrimenti Lazy per le performance di caricamento

E NLTK come gestisce tutto questo?

Architettura di Caricamento
Mentre spaCy è un pacchetto completo, NLTK è un banco di lavoro artigianale
In NLTK scarichiamo i singoli strumenti necessari.
Volete dividere in frasi, scaricate "punkt". Questo approccio ci da un controllo chirurgico ma richiede più passaggi manuali.

Il punto cruciale è la compatibilià delle versioni, assicurati che la versione della libreria e la version del modello siano allineati, o rischi che il cervello parli un dialetto che il software non capisce più.

Ma quanto ci costa tutto questo in termini di performace?

Costo Computazionale del Caricamento
Analisi dell'impatto in memoria
Il tempo di inizializzazione della pipeline è dominato dal caricamento dei tensori in memoria. Per un modello small, la complessità temporale di avvio è trascurabile, ma per modelli large con word vectors estesi, l'impatto sulla RAM è lineare rispetto alla dimensione del dizionario.
Il tempo di caricamente cresce linearmente con la dimensione del vocabolario 
Possiamo modellare il tempo di caricamento come una funzione del numero di parametri e della larghezza della matrice di embedding.

Contronto Prestazioni e Precisione
Mattere alla prova i due giganti
NLTK e spaCy rappresentano due filosofie diverse: il primo è un coltellino svizzero accademico (ha mille funzioni, è trasparente perfetto per chi fa ricerca), il secondo è una catena di montaggio industriale (ottimizzato per la velocità).
Valutare quale usare dipende dal trade-off tra la precisione della scomposizione e il numero di documenti da processare al secondo

Throughput vs Accuracy
Dimensionare il preprocessing
- Pipeline Latency: spaCy è sensibilmente più veloce grazie al core scritto in Cython e all'elaborazione a singolo passaggio (one-pass processing)
- Precisione dei Confini: NLTK permette di regolare manualmente le euristiche di splitting, risultando talvolta superiore in testo molto destrutturati
- Robustessa Linguistica: spaVy gestisce meglio le dipendenze contestuali (es. punti nelle abbreviazioni) grazie ai modelli statistici incorporati
- Multithreading: spaCy offre il metodo 'nlp.pipe' per processare stram di testo in parallelo, ottimizzando l'uso di moderni processori multi-core

Vediamo come questo si applica ai casi reali.

Scelta del Tool in base al Dataset
Se dobbiamo processare gigabyte di log di un server o tweet, spaCy è la scelta obbligato per via dell'efficienza nel throughput dei token
NLTK è preferibile quando l'obbiettivo è la trasparenza algoritmica, ovvero quando vogliamo testare variazioni minime nelle regole di segmentazione
Inotre se lavori con il Deep Learning SpaCy ha già pronti i ponti perfetto per PyTorch o TensorFlow tramite una classe "DocBin" che facilita il passaggio dei token ai tensori pronti per la GPU

Metriche di Precisione
Quantificare l'errore di segmentazione
Per misurare la precisione di un tokenizer, confrontiamo i token generati con un set di "Golden Token" annotati da esperti umani. Uilizziamo la metrica di Precisione per valutare quanti token identificati siano effettivamente corretti
Un tokenizer perfetto minimizza i falsi positivi (token spezzati male) e i falsi negati (token fusi erroneamente)

Spesso però i modelli standard falliscono 
Vediamo come rimediare

Personalizzazione dei Domini Specifici.
Adattare il tokenizer al contesto
Ogni dominio professionale ha il suo "gergo" tecnico che sfida le regole standard della lingua. Nel dominio legale, ad esempio, sigle come "art.12" o "c.p.c." non devono essere divisi dal punto, altrimenti perdiamo il riferimento normativo.
In questi casi dobbiamo iniettare regole personalizzate nelle pipeline per proteggere termini speciali e garantire che il modello a valle riceva unità semantiche intatte.

Ma quali sono gli strumenti per farlo?

Eccesioni e Regole
Proteggere il significato tecnico.
- Special Case Rules: sono come dei post-it che attacchiamo sul manuale del tokenizer. Quando vedi questa parola, non toccarla. L'aggiunta di entry nel dizionario del tokenizer per definire come stringhe specifiche debbano essere scomposte o preservate.
- Infix e Prefix Regex: possiamo anche modificare le espressioni regolare, regez, che definiscono i prefissi ed i suffissi
- Attriute Ruler: lui è ancora più potente, ci permette di dire che se una parola segue un certo pattern deve essere marchiata in un modo specifico a prescindere da quello che dice il modello statistico
- Pipeline Components: creazione di classi Python custom da inserire tra il tokenizer e il parser per trasformazioni on-the-fky

Applichiamo tutto questo ad un caso concreto, il dominio legale

Caso Studio: Dominio Legale
Configurare il tokenizer per trattare i riferimenti legislativi come singoli token aiuta il modello di Named Entity Recognition a identificarli come entità unitario (Gestione riferimento)
In ambiti tecnici, operatori o formula semplici (es. "CO2" o "x>y") richiedono regole di infix personalizzate per non essere trattate come puro rumore sintattico
E' best practice centralizzare queste regole in un file di configurazione separato, permettendo di aggiornare il "vocabolario tecnico" senza riscrivere il codice di pre-processing, il giurista deve solo aggionrare con le nuove sigle mentre lo sviluppatore deve aggiornare il proprio vocabolario.

La tokenizzazione personalizzata può essere descritta come l'estensione dell'insieme dei simboli terminali di una grammatia formale. Definiamo set di eccezioni E che ha precedenza sulle regole di splitting stadard S
Il tokenizer applica un mapping prioritario: se la sottostringa appartiene a E, viene restituita come token singolo, altrimenti si procede con l'applicazione di S.

Con la tokenizzazione personalizzata non mi accontento delle regole standard del tokenizer, ma decido come certi elementi del testo devono essere separati o mantenuti uniti.
E' utile quando il testo contiene elementi specifici del tuo dominio, per esempoi
ord-45872 Rossi srl art pav1.1313.002 prezzo 1.250,5
un tokenizer generico potrebbe spezzare  ord e 45872 ma voglio che rimangano uniti


In [2]:
import os  # Importiamo il modulo per gestire i comandi di sistema (es. download modelli)
import time # Importiamo time per calcolare la latenza dei diversi approcci
import nltk # Libreria storica per il processamento del linguaggio naturale
import spacy # Framework industriale moderno ottimizzato per le performance
from spacy.symbols import ORTH # Importiamo il simbolo ORTH per definire regole ortografiche custom

def run_nlp_benchmark():
    # Definiamo un testo complesso con abbreviazioni legali e punteggiatura ambigua
    # La sfida è non dividere abbreviazioni come "c.p.c." o "D.Lgs."
    text = "L'art. 24 del c.p.c. definisce i termini per l'appello nel D.Lgs. 231/01."

    print(f"--- ANALISI DEL TESTO: {text} ---\n")

    # --- 1. APPROCCIO NLTK (Granulare/Regolistico) ---
    # Scarichiamo la risorsa 'punkt_tab', necessaria nelle versioni 2026 per la tokenizzazione
    nltk.download('punkt_tab', quiet=True) 
    
    # Registriamo il tempo di inizio per il benchmark NLTK
    start_time = time.time()
    
    # Tokenizzazione NLTK: usa l'algoritmo 'Punkt' per decidere dove spezzare le parole
    nltk_tokens = nltk.word_tokenize(text, language='italian')
    
    # Calcoliamo la durata dell'operazione
    nltk_duration = time.time() - start_time
    
    # Stampiamo i risultati e il tempo impiegato
    print(f"[NLTK] Token trovati ({len(nltk_tokens)}): {nltk_tokens}")
    print(f"[NLTK] Tempo di esecuzione: {nltk_duration:.6f}s\n")

    # --- 2. APPROCCIO SPACY (Industriale/Statistico) ---
    try:
        # Carichiamo il modello italiano 'small' pre-addestrato
        nlp = spacy.load("it_core_news_sm")
    except OSError:
        # Se il modello non è installato, eseguiamo il download via terminale
        os.system("python -m spacy download it_core_news_sm")
        # Ricarichiamo il modello dopo l'installazione
        nlp = spacy.load("it_core_news_sm")

    # Registriamo il tempo di inizio per il benchmark spaCy
    start_time = time.time()
    
    # Creiamo l'oggetto 'Doc': spaCy analizza il testo in un unico passaggio (pipeline)
    doc = nlp(text)
    
    # Estraiamo il testo di ogni token dall'oggetto Doc
    spacy_tokens = [token.text for token in doc]
    
    # Calcoliamo la durata dell'operazione
    spacy_duration = time.time() - start_time
    
    # Stampiamo i risultati: noterai che spaCy gestisce meglio gli apostrofi rispetto a NLTK
    print(f"[spaCy] Token trovati ({len(spacy_tokens)}): {spacy_tokens}")
    print(f"[spaCy] Tempo di esecuzione: {spacy_duration:.6f}s\n")

    # --- 3. PERSONALIZZAZIONE DEL TOKENIZER (Dominio Legale) ---
    # Definiamo la regola: quando trovi "c.p.c.", non dividerlo in "c.", "p.", "c."
    # ORTH definisce la stringa esatta che deve essere restituita come unico token
    special_case = [{ORTH: "c.p.c."}]
    
    # Aggiungiamo il caso speciale alla logica del tokenizer di spaCy
    nlp.tokenizer.add_special_case("c.p.c.", special_case)
    
    # Rieseguiamo l'analisi sullo stesso testo
    doc_custom = nlp(text)
    
    # Estraiamo nuovamente i token per verificare la modifica
    custom_tokens = [t.text for t in doc_custom]
    
    # Mostriamo come il numero di token sia diminuito poiché l'abbreviazione è ora unita
    print(f"[spaCy Custom] Dopo la regola 'c.p.c.': {custom_tokens}")
    print(f"[spaCy Custom] Nuova lunghezza: {len(custom_tokens)}")
    print("Nota: 'c.p.c.' è ora trattato come un'unica entità semantica.\n")

    # --- 4. ANALISI DEI METADATI (Deep Insight) ---
    # Mostriamo informazioni utili per il debugging dei modelli NLP
    print("[Insight] Analisi della struttura dei primi 5 token:")
    for token in list(doc_custom)[:5]:
        # Stampa: Testo | Is_Punct (è punteggiatura?) | Is_Abbrev (è abbreviazione?)
        print(f"Token: {token.text:12} | Punct: {str(token.is_punct):5} | Like_Num: {token.like_num}")

# Avviamo il benchmark se lo script viene eseguito direttamente
if __name__ == "__main__":
    run_nlp_benchmark()

--- ANALISI DEL TESTO: L'art. 24 del c.p.c. definisce i termini per l'appello nel D.Lgs. 231/01. ---

[NLTK] Token trovati (16): ["L'art", '.', '24', 'del', 'c.p.c', '.', 'definisce', 'i', 'termini', 'per', "l'appello", 'nel', 'D.Lgs', '.', '231/01', '.']
[NLTK] Tempo di esecuzione: 0.447739s

[spaCy] Token trovati (17): ["L'", 'art.', '24', 'del', 'c.p.c', '.', 'definisce', 'i', 'termini', 'per', "l'", 'appello', 'nel', 'D.Lgs', '.', '231/01', '.']
[spaCy] Tempo di esecuzione: 0.008282s

[spaCy Custom] Dopo la regola 'c.p.c.': ["L'", 'art.', '24', 'del', 'c.p.c.', 'definisce', 'i', 'termini', 'per', "l'", 'appello', 'nel', 'D.Lgs', '.', '231/01', '.']
[spaCy Custom] Nuova lunghezza: 16
Nota: 'c.p.c.' è ora trattato come un'unica entità semantica.

[Insight] Analisi della struttura dei primi 5 token:
Token: L'           | Punct: False | Like_Num: False
Token: art.         | Punct: False | Like_Num: False
Token: 24           | Punct: False | Like_Num: True
Token: del          | Punct: F